In [ ]:
import marimo as mo

# Imports

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

interruption: This cell was interrupted and needs to be re-run

In [ ]:
import gpflow
import trieste
from trieste.space import Box
from trieste.models.gpflow.models import GaussianProcessRegression

2026-04-22 14:59:44.627125: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-22 14:59:44.793076: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9373] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-22 14:59:44.793290: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-22 14:59:44.818917: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1534] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-22 14:59:44.878604: I tensorflow/core/platform/cpu_feature_guar

interruption: This cell was interrupted and needs to be re-run

In [ ]:
# group sequential design assessment imports
from py_group_sequential_designs import generate_boundaries as bd
from py_group_sequential_designs import feasibility_penalty as fp
from py_group_sequential_designs import boundary_manipulations as fmt_bd
from py_group_sequential_designs import function_to_minimize as fn_min
from py_group_sequential_designs import generate_gpr_input as gen_input
from py_group_sequential_designs import simulate as sim
from py_group_sequential_designs import sample_size as ss

# Trial design settings

In [ ]:
num_analyses = 3
target_alpha = 0.05
target_power = 0.9
delta0 = 0
delta1 = 1.0
sigma2 = 3.0

mu = ss.sample_size_means(
    ratio=1,
    variance=sigma2,
    power=target_power,
    alpha=target_alpha,
    delta=delta1
)
print(f"Single-stage sample size mu = {mu:.2f}")

Single-stage sample size mu = 154.15


interruption: This cell was interrupted and needs to be re-run

# Reverse parameterization

BO vector: $(c, \Delta u_3, \Delta l_3, \Delta u_2, \Delta l_2)$

In [ ]:
def reverse_to_boundaries(params, K):
    params = np.asarray(params).flatten()
    c = params[0]

    delta_u = params[1::2][::-1]
    delta_l = params[2::2][::-1]

    upper_bounds = np.array([c + np.sum(delta_u[k:]) for k in range(K)])
    lower_bounds = np.array([c - np.sum(delta_l[k:]) for k in range(K)])

    return upper_bounds, lower_bounds

def boundaries_to_reverse(upper_bounds, lower_bounds):
    upper_bounds = np.asarray(upper_bounds)
    lower_bounds = np.asarray(lower_bounds)

    K = len(upper_bounds)
    c = upper_bounds[-1]

    delta_u = np.diff(upper_bounds[::-1])
    delta_l = np.diff(lower_bounds)[::-1]

    increments = np.empty(2 * (K - 1))
    increments[0::2] = delta_u
    increments[1::2] = delta_l

    return np.concatenate([[c], increments])

interruption: This cell was interrupted and needs to be re-run

# Objective function

In [ ]:
def obj_f(
        mu,
        params,
        n_analyses,
        target_power,
        target_alpha):

    params = np.array(params)
    upper_bounds, lower_bounds = reverse_to_boundaries(params = params, K = n_analyses)

    n_power09, calc_power = ss.find_sample_size(
        power_target = target_power,
        n_analyses = n_analyses,
        upper_bounds = upper_bounds,
        lower_bounds = lower_bounds,
        null_hypothesis = delta0,
        alt_hypothesis = delta1,
        variance = sigma2
    )

    beta_prime = 1-calc_power

    alpha_prime = sim.group_sequential_designs(
        n_analyses = num_analyses,
        upper_bounds = upper_bounds,
        lower_bounds = lower_bounds,
        n_patients = n_power09, 
        null_hypothesis = delta0,
        alt_hypothesis = delta1,
        variance = sigma2
    )[1]

    max_ess = ss.max_ess(
        n_analyses = n_analyses,
        upper_bounds = upper_bounds,
        lower_bounds = lower_bounds,
        n_patients = n_power09
    )

    penalty = fp.smooth_penalty(
        mu = mu,
        power = target_power,
        alpha = target_alpha,
        beta_prime = beta_prime,
        alpha_prime = alpha_prime
    )

    f_val = fn_min.function_to_minimize(max_ess_val=max_ess/mu, penalty=penalty)

    return (
        alpha_prime,
        calc_power,
        n_power09,
        max_ess,
        f_val
    )

interruption: This cell was interrupted and needs to be re-run

# Objective function value - triangular

In [ ]:
tri = bd.calculate_triangular_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    delta=delta1,
    n_patients=20
)

tri

interruption: This cell was interrupted and needs to be re-run

In [ ]:
tri_params = boundaries_to_reverse(lower_bounds = tri[1], upper_bounds = tri[0])

_,_,_,_,tri_obj = obj_f(
    mu = mu,
    params = tri_params,
    n_analyses = num_analyses,
    target_power = target_power,
    target_alpha = target_alpha
)

tri_params = boundaries_to_reverse(tri[0], tri[1])
c0 = tri_params[0]

print(f"Triangular benchmark objective: {tri_obj:.4f}")
print(f"Original trriangular params: {np.round(np.concatenate((tri[0], tri[1])), 4)}")
print(f"Reparameterized triangular params: {np.round(tri_params, 4)}")
print(f"Meeting point c0 = {c0:.4f}")

Triangular benchmark objective: 0.2704
Original trriangular params: [2.1196 1.8735 1.8356 0.     1.1241 1.8356]
Reparameterized triangular params: [1.8356 0.0379 0.7115 0.2461 1.1241]
Meeting point c0 = 1.8356


interruption: This cell was interrupted and needs to be re-run

# Search space

In [ ]:
lower = np.array([c0 - 3.0, 0.0, 0.0, 0.0, 0.0])
upper = np.array([c0 + 3.0, 4.0, 4.0, 4.0, 4.0])

search_space = Box(lower=lower, upper=upper)
print(f"  lower: {np.round(search_space.lower.numpy(), 3)}")
print(f"  upper: {np.round(search_space.upper.numpy(), 3)}")

  lower: [-1.164  0.     0.     0.     0.   ]
  upper: [4.836 4.    4.    4.    4.   ]


Ancestor raised: An ancestor raised an exception (KeyboardInterrupt)

# Initialisation

In [ ]:
_poc = bd.calculate_pocock_boundaries(
    n_analyses=num_analyses, alpha=0.05, n_patients=20
)
poc_params = boundaries_to_reverse(_poc[0], _poc[1])

_obf = bd.calculate_of_boundaries(
    n_analyses=num_analyses, alpha=0.05, n_patients=20
)
obf_params = boundaries_to_reverse(_obf[0], _obf[1])

interruption: This cell was interrupted and needs to be re-run

In [ ]:
_,_,_,_,poc_obj_f = obj_f(
    mu = mu, 
    params = poc_params,
    n_analyses = num_analyses,
    target_power = target_power,
    target_alpha = target_alpha
)

_,_,_,_,obf_obj_f = obj_f(
    mu = mu, 
    params = obf_params,
    n_analyses = num_analyses,
    target_power = target_power,
    target_alpha = target_alpha
)

interruption: This cell was interrupted and needs to be re-run

In [ ]:
design_matrix = np.concatenate((np.atleast_2d(poc_params), np.atleast_2d(obf_params)))
output_vals = np.concatenate((np.atleast_2d(poc_obj_f), np.atleast_2d(obf_obj_f)))

interruption: This cell was interrupted and needs to be re-run

In [ ]:
initial_data = trieste.data.Dataset(
    query_points = design_matrix,
    observations = output_vals
)

Ancestor raised: An ancestor raised an exception (KeyboardInterrupt)

In [ ]:
print(f"Initial dataset:\n{design_matrix}\n")
print(f"Initial f(x):\n{output_vals}")

Initial dataset:
[[1.99218256 0.         3.98436511 0.         0.        ]
 [1.70967102 0.38423979 3.80358183 0.86732626 0.86732626]]

Initial f(x):
[[0.36749402]
 [0.33122993]]


# GP model

In [ ]:
_kernel = gpflow.kernels.Matern52(
    lengthscales=[1.0] * design_matrix.shape[1]
)

_gpr = gpflow.models.GPR(
    data      = (design_matrix, output_vals),
    kernel    = _kernel,
    likelihood = gpflow.likelihoods.Gaussian()
)

gpflow.utilities.print_summary(_gpr, fmt="notebook")
bayes_opt_model = GaussianProcessRegression(_gpr)

2026-04-21 22:34:37.164999: W external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:225] Falling back to the CUDA driver for PTX compilation; ptxas does not support CC 12.0
2026-04-21 22:34:37.165010: W external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:228] Used ptxas at ptxas
2026-04-21 22:34:37.165036: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-21 22:34:37.229063: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-21 22:34:37.229448: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-21 22:34:37.229468: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191]

Ancestor raised: An ancestor raised an exception (KeyboardInterrupt)

# Bayesian optimisation loop

In [ ]:
ask_tell = trieste.ask_tell_optimization.AskTellOptimizer(
    search_space     = search_space,
    datasets         = initial_data,
    models           = bayes_opt_model,
    acquisition_rule = trieste.acquisition.rule.EfficientGlobalOptimization(
        optimizer = trieste.acquisition.optimizer.generate_continuous_optimizer(
            num_optimization_runs = 500
        )
    )
)

2026-04-21 22:34:45.267696: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-21 22:34:45.436950: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-21 22:34:47.542121: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.


Ancestor raised: An ancestor raised an exception (KeyboardInterrupt)

In [ ]:
num_repeats   = 1000
when_to_print = 100
n_design_goal_met = 0
design_goal_met_list = []

epsilon = 0.015
n_within_epsilon = 0
within_epsilon_list = []

# start a timer
start_time = time.time()

for _i in range(num_repeats):
    x_new = ask_tell.ask()

    alpha_new, power_new, n_power_09_new, max_ess_new, y_new = obj_f(
        mu = mu, 
        params = x_new,
        n_analyses = num_analyses,
        target_power = target_power,
        target_alpha = target_alpha
    )

    # how many designs meet alpha 0.05 and power 0.9
    design_goal_met = (alpha_new <= target_alpha) & (power_new >= target_power - 0.05)
    if design_goal_met:
        n_design_goal_met += 1
    design_goal_met_list.append(design_goal_met)

    # how many designs are within epsilon of alpha and power
    within_epsilon = ( (alpha_new <= (target_alpha + epsilon)) & (alpha_new >= (target_alpha - epsilon)) )
    if within_epsilon:
        n_within_epsilon += 1
    within_epsilon_list.append(within_epsilon)

    ask_tell.tell(trieste.data.Dataset(
        query_points = x_new,
        observations = np.array([[y_new]])
    ))

    if (_i + 1) % when_to_print == 0:
        print(
            f"\nLoop {_i+1} completed. "
            f"Feasible: {n_design_goal_met}/{_i+1} "
            f"({100*n_design_goal_met/(_i+1):.0f}%).",
            end=""
         )
    elif (_i > when_to_print) and ((_i + 1) % 5 == 0):
        print(".", end="")

# end the timer
end_time = time.time()
execution_time = end_time - start_time

print(f"\nDone. Feasible BO proposals: {n_design_goal_met}/{num_repeats}")

2026-04-21 22:34:53.305169: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-21 22:34:53.305547: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-21 22:34:53.305559: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-21 22:34:53.305569: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-21 22:34:53.306070: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-21


Loop 100 completed. Feasible: 51/100 (51%)........

............
Loop 200 completed. Feasible: 101/200 (50%)....................
Loop 300 completed. Feasible: 158/300 (53%)....................
Loop 400 completed. Feasible: 221/400 (55%)....................
Loop 500 completed. Feasible: 274/500 (55%)....................
Loop 600 completed. Feasible: 330/600 (55%)....................
Loop 700 completed. Feasible: 376/700 (54%)....................
Loop 800 completed. Feasible: 427/800 (53%)....................
Loop 900 completed. Feasible: 475/900 (53%)....................
Loop 1000 completed. Feasible: 524/1000 (52%).
Done. Feasible BO proposals: 524/1000


Ancestor raised: An ancestor raised an exception (KeyboardInterrupt)

In [ ]:
best_obj_f = np.min(ask_tell.to_result().try_get_final_dataset().observations[2:])

best_idx = np.argmin(ask_tell.to_result().try_get_final_dataset().observations[2:])

best_bounds = reverse_to_boundaries(
    ask_tell.to_result().try_get_final_dataset().query_points[2:][best_idx], 
    K=num_analyses
)

best_n09, _ = ss.find_sample_size(
    power_target = target_power,
    n_analyses = num_analyses,
    upper_bounds = best_bounds[0],
    lower_bounds = best_bounds[1],
    null_hypothesis = delta0,
    alt_hypothesis = delta1,
    variance = sigma2
)

print(f"Bayesian optim:       {num_repeats} evaluations")
print(f"Loop took:            {execution_time/60:.1f} min")
print(f"Best objective val:   {best_obj_f}")
print(f"Best objective idx:   {best_idx}")
print(f"Feasible epsilon:     {n_within_epsilon}/{num_repeats} ({100*np.mean(within_epsilon_list):.1f}%)")
print(f"Best n w/power 0.9:   {best_n09}")
print(f"Better than tri?      {best_obj_f < tri_obj}")

Bayesian optim:       1000 evaluations
Loop took:            19.1 min
Best objective val:   0.28885703196418777
Best objective idx:   292
Feasible epsilon:     43/1000 (4.3%)
Best n w/power 0.9:   18.472900390625
Better than tri?      False


Ancestor raised: An ancestor raised an exception (KeyboardInterrupt)

In [ ]:
best_bounds

# Best trial design properties

In [ ]:
sim.group_sequential_designs(
    n_analyses = num_analyses,
    upper_bounds = best_bounds[0],
    lower_bounds = best_bounds[1],
    n_patients = best_n09,
    null_hypothesis = delta0,
    alt_hypothesis = delta1,
    variance = sigma2
)

# Run Bayes opt 1000 times

In [ ]:
# how many loops of n_baseline to run
n_experiments = 50
n_loops = 1000

# collect some important values
loop_observations = []
loop_query_points = []
loop_design_goal_met = []
loop_within_epsilon = []
loop_best_idx = []
loop_best_obj_f = []
loop_best_bounds = []
loop_execution_time = []
loop_best_n09 = []

for _j in range(n_experiments):

    when_to_print_loop = 100
    n_design_goal_met_loop = 0
    design_goal_met_list_loop = []

    epsilon_loop = 0.015
    n_within_epsilon_loop = 0
    within_epsilon_list_loop = []

    # start a timer
    start_time_loop = time.time()

    # reset the ask_tell interface every experiment
    loop_ask_tell = trieste.ask_tell_optimization.AskTellOptimizer(
        search_space     = search_space,
        datasets         = initial_data,
        models           = bayes_opt_model,
        acquisition_rule = trieste.acquisition.rule.EfficientGlobalOptimization(
            optimizer = trieste.acquisition.optimizer.generate_continuous_optimizer(
                num_optimization_runs = 500
            )
        )
    )

    for _i in range(n_loops):
        x_new_loop = loop_ask_tell.ask()

        alpha_new_loop, power_new_loop, n_power_09_new_loop, max_ess_new_loop, y_new_loop = obj_f(
            mu = mu, 
            params = x_new_loop,
            n_analyses = num_analyses,
            target_power = target_power,
            target_alpha = target_alpha
        )

        # how many designs meet alpha 0.05 and power 0.9
        design_goal_met_loop = (alpha_new_loop <= target_alpha) & (power_new_loop >= target_power - 0.05)
        if design_goal_met_loop:
            n_design_goal_met_loop += 1
        design_goal_met_list_loop.append(design_goal_met_loop)

        # how many designs are within epsilon of alpha and power
        alpha_target_low = alpha_new_loop <= (target_alpha + epsilon_loop)
        alpha_target_high = alpha_new_loop >= (target_alpha - epsilon_loop)
        within_epsilon_loop = ( alpha_target_low & alpha_target_high )
        if within_epsilon_loop:
            n_within_epsilon_loop += 1
        within_epsilon_list_loop.append(within_epsilon_loop)

        loop_ask_tell.tell(trieste.data.Dataset(
            query_points = x_new_loop,
            observations = np.array([[y_new_loop]])
        ))

        if (_i + 1) % when_to_print_loop == 0:
            print(
                f"\nLoop {_i+1} completed. "
                f"Feasible: {n_design_goal_met_loop}/{_i+1} "
                f"({100*n_design_goal_met_loop/(_i+1):.0f}%).",
                end=""
             )
        elif (_i > when_to_print_loop) and ((_i + 1) % 5 == 0):
            print(".", end="")

    # end the timer
    end_time_loop = time.time()
    execution_time_loop = end_time_loop - start_time_loop

    # save helpful items from loop
    loop_observations.append(
        np.array(loop_ask_tell.to_result().try_get_final_dataset().observations[2:]).flatten()
    )

    loop_query_points.append(
        np.array(loop_ask_tell.to_result().try_get_final_dataset().query_points[2:])
    )

    loop_design_goal_met.append(design_goal_met_list_loop)
    loop_within_epsilon.append(within_epsilon_list_loop)

    print("=============================")
    print(f"= Completed experiment {_j+1}. =")
    print("=============================")

    best_obj_f_loop = np.min(loop_ask_tell.to_result().try_get_final_dataset().observations[2:])

    best_idx_loop = np.argmin(loop_ask_tell.to_result().try_get_final_dataset().observations[2:])

    best_bounds_loop = reverse_to_boundaries(
        loop_ask_tell.to_result().try_get_final_dataset().query_points[2:][best_idx_loop], 
        K=num_analyses
    )

    best_n09_loop, _ = ss.find_sample_size(
        power_target = target_power,
        n_analyses = num_analyses,
        upper_bounds = best_bounds_loop[0],
        lower_bounds = best_bounds_loop[1],
        null_hypothesis = delta0,
        alt_hypothesis = delta1,
        variance = sigma2
    )

    loop_best_obj_f.append(best_obj_f_loop)
    loop_best_idx.append(best_idx_loop)
    loop_best_bounds.append(best_bounds_loop)
    loop_execution_time.append(execution_time_loop)
    loop_best_n09.append(best_n09_loop)

    print(f"Bayesian optim:       {n_loops} evaluations")
    print(f"Loop took:            {execution_time_loop/60:.1f} min")
    print(f"Feasible BO:          {n_design_goal_met_loop}/{n_loops} ({100*np.mean(design_goal_met_loop):.1f}%)")
    print(f"Feasible epsilon:     {n_within_epsilon_loop}/{n_loops} ({100*np.mean(within_epsilon_list_loop):.1f}%)")
    print(f"Best objective val:   {best_obj_f_loop}")
    print(f"Best objective idx:   {best_idx_loop}")
    print(f"Best n w/power 0.9:   {best_n09_loop}")
    print(f"Better than tri?      {best_obj_f_loop < tri_obj}\n")


Loop 100 completed. Feasible: 58/100 (58%)....................
Loop 200 completed. Feasible: 113/200 (56%)....................
Loop 300 completed. Feasible: 160/300 (53%)....................
Loop 400 completed. Feasible: 210/400 (52%)....................
Loop 500 completed. Feasible: 269/500 (54%)....................
Loop 600 completed. Feasible: 319/600 (53%)....................
Loop 700 completed. Feasible: 378/700 (54%)....................
Loop 800 completed. Feasible: 431/800 (54%)....................
Loop 900 completed. Feasible: 482/900 (54%)....................
Loop 1000 completed. Feasible: 536/1000 (54%).=============================
= Completed experiment 1. =
Bayesian optim:       1000 evaluations
Loop took:            19.0 min
Feasible BO:          536/1000 (100.0%)
Feasible epsilon:     43/1000 (4.3%)
Best objective val:   0.2661185760994103
Best objective idx:   799
Best n w/power 0.9:   17.252685546875
Better than tri?      True


Loop 100 completed. Feasible: 46/100 (4

Ancestor raised: An ancestor raised an exception (KeyboardInterrupt)

# Best bounds characteristics

In [ ]:
for (_i, _bound) in enumerate(loop_best_bounds):
    _chars = sim.group_sequential_designs(
        n_analyses = num_analyses,
        upper_bounds = _bound[0],
        lower_bounds = _bound[1],
        n_patients = best_n09,
        null_hypothesis = delta0,
        alt_hypothesis = delta1,
        variance = sigma2
    )

    print(f"Run: {_i+1}")
    print(f"Upper: {_bound[0]}")
    print(f"Lower: {_bound[1]}")
    print(f"Alpha: {_chars[1]}")
    print(f"Beta:  {_chars[2]}")
    print(f"ESS:   {_chars[3]}\n")

Ancestor raised: An ancestor raised an exception (KeyboardInterrupt)